# Notebook 3: throw metrics

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#This imports all of our necessary libraries

In [2]:
df = pd.read_csv("/Users/arnavchouksey/kinetisense/data/synthetic/synthetic_1000hz.csv")
df.head() #Our practice DATABASE (CSV) that we will be using for this

,time_s,gx,gy,gz,ax,ay,az
0,0.000,0.314700,-1.607008,1.084785,0.002820,0.012064,0.981732
1,0.001,4.302834,3.144411,-1.862668,-0.014538,0.021140,1.019863
2,0.002,-3.014653,-1.263530,0.621019,-0.010477,0.003275,1.018717
3,0.003,-5.802815,0.252957,-2.992260,0.048790,0.017633,0.983330
4,0.004,-0.634025,-0.420535,0.044897,0.004248,-0.002296,1.021145


In [3]:
events = pd.read_csv("/Users/arnavchouksey/kinetisense/data/synthetic/synthetic_events.csv")

In [4]:
def gyro_magnitude(table):
    return np.sqrt(table["gx"] ** 2 + table["gy"] ** 2 + table["gz"] ** 2)
    #Define a new function called Gyro_magnetude which is similar to the gyro_mag from the last two notebooks

In [5]:
gyro_magnitude(df).max() #Maximum Gyro magnitude

np.float64(4901.206574580212)

In [6]:
gyro_magnitude(df).idxmax() #What 'id', number the gyro magnitude was at

27121

In [7]:
gyro_magnitude(df) #Comparing Gyro_mag with everything the df 'database'

0        1.964248
1        5.645461
2        3.327207
3        6.533779
4        0.762138
           ...   
59995    2.486520
59996    7.102971
59997    3.056473
59998    4.761211
59999    4.929092
Length: 60000, dtype: float64

In [8]:
df["window"] = (df["time_s"] // 1).astype(int) #Create a time 'window' for a throw
df["gyro_mag"] = gyro_magnitude(df) #Add a new column for each gyromagnitude in gf
peaks = df.groupby("window")["gyro_mag"].max() #Groups by windows and then goes deeper by labeling the gyromagnetute in that window added up

THRESHOLD = 1500          # provisional, from Notebook 2
detected = peaks[peaks > THRESHOLD]
detected #THIS IS THE DETECTED TABLE with everything that exceeds the threshold HERE!

window
5     4338.251844
11    2149.529838
18    2981.568587
27    4901.206575
33    4080.434324
39    3134.096190
47    4537.185041
54    4120.485037
Name: gyro_mag, dtype: float64

In [9]:
peaks #These are all of the peaks for every single window!

window
0       52.875551
1       52.783543
2       54.073579
3       53.350789
4       53.367132
5     4338.251844
6       52.811551
7       53.797900
8      254.018304
9      399.781163
10      54.607373
11    2149.529838
12      55.389873
13      53.380920
14      53.357972
15     394.365528
16      54.882603
17      54.259157
18    2981.568587
19      52.837571
20      53.764978
21      54.109777
22     522.589565
23      95.204371
24      57.161322
25      53.916229
26      54.334685
27    4901.206575
28      56.977900
29      54.977719
30     693.874968
31     476.208704
32      52.278407
33    4080.434324
34      54.184177
35      54.741645
36     298.118716
37     669.139937
38      53.796586
39    3134.096190
40      54.673223
41      54.043919
42      53.792851
43     551.602484
44      51.757335
45      52.799217
46      53.361249
47    4537.185041
48      53.939518
49      55.443838
50      52.736301
51     507.284595
52      54.277647
53      53.944706
54    4120.485037
55 

In [10]:
df #df with NEW columns added to it!

,time_s,gx,gy,gz,ax,ay,az,window,gyro_mag
0,0.000,0.314700,-1.607008,1.084785,0.002820,0.012064,0.981732,0,1.964248
1,0.001,4.302834,3.144411,-1.862668,-0.014538,0.021140,1.019863,0,5.645461
2,0.002,-3.014653,-1.263530,0.621019,-0.010477,0.003275,1.018717,0,3.327207
3,0.003,-5.802815,0.252957,-2.992260,0.048790,0.017633,0.983330,0,6.533779
4,0.004,-0.634025,-0.420535,0.044897,0.004248,-0.002296,1.021145,0,0.762138
...,...,...,...,...,...,...,...,...,...
59995,59.995,-0.744222,-1.279341,1.998049,0.011971,-0.025193,0.972145,59,2.486520
59996,59.996,-5.404739,-2.321580,-3.981364,-0.023924,-0.024352,0.983035,59,7.102971
59997,59.997,-0.528172,2.480683,1.705658,-0.012265,0.020655,0.999053,59,3.056473
59998,59.998,-4.696312,-0.746099,-0.238987,-0.036442,-0.005642,1.003420,59,4.761211


In [11]:
one = df[df["window"] == 27]
one["gyro_mag"].max(), one["gyro_mag"].idxmax()

(np.float64(4901.206574580212), 27121)

In [12]:
axes = ["gx", "gy", "gz"]
one = df[df["window"] == 27]
one[axes].abs().max()

gx    1948.815853
gy    3030.820759
gz    3322.358828
dtype: float64

In [13]:
def measure_window(table, window):
    rows = table[table["window"] == window]
    peak_row = rows["gyro_mag"].idxmax()
    return {
        "peak_time_s": table.loc[peak_row, "time_s"],
        "peak_dps": rows["gyro_mag"].max(),
        "clipped": bool((rows[["gx", "gy", "gz"]].abs() >= 3999.99).any().any()),
    }

In [14]:
measure_window(df, 27) #Test out our function 


{'peak_time_s': np.float64(27.121),
 'peak_dps': np.float64(4901.206574580212),
 'clipped': False}

In [15]:
measure_window(df, 5) #Test out our funciton

{'peak_time_s': np.float64(5.394),
 'peak_dps': np.float64(4338.25184445237),
 'clipped': True}

In [16]:
events #Other table we imported in the start

,event_time_s,label,true_peak_dps
0,5.393809,throw,5021.216869
1,9.143809,not_throw,385.504934
2,11.928604,throw,2157.502763
3,15.678604,not_throw,383.712589
4,18.971310,throw,2974.022361
5,22.721310,not_throw,512.227985
6,27.120936,throw,4907.067178
7,30.870936,not_throw,702.249023
8,33.459940,throw,5047.055059
9,37.209940,not_throw,694.600190


In [17]:
throws= events[events["label"]=="throw"]
swings= events[events["label"]=="not_throw"]

In [18]:
throws

,event_time_s,label,true_peak_dps
0,5.393809,throw,5021.216869
2,11.928604,throw,2157.502763
4,18.971310,throw,2974.022361
6,27.120936,throw,4907.067178
8,33.459940,throw,5047.055059
10,39.861702,throw,3144.810593
12,47.738363,throw,4555.785182
14,54.170801,throw,4119.960431


In [19]:
results=[]
for w in detected.index:
    results.append(measure_window(df, w))
throws_table = pd.DataFrame(results)
throws_table.insert(0, "throw_id", range(1, len(throws_table) + 1))
throws_table

,throw_id,peak_time_s,peak_dps,clipped
0,1,5.394,4338.251844,True
1,2,11.929,2149.529838,False
2,3,18.971,2981.568587,False
3,4,27.121,4901.206575,False
4,5,33.459,4080.434324,True
5,6,39.862,3134.096190,False
6,7,47.738,4537.185041,False
7,8,54.171,4120.485037,False


In [20]:
detected

window
5     4338.251844
11    2149.529838
18    2981.568587
27    4901.206575
33    4080.434324
39    3134.096190
47    4537.185041
54    4120.485037
Name: gyro_mag, dtype: float64

In [21]:
throws_table

,throw_id,peak_time_s,peak_dps,clipped
0,1,5.394,4338.251844,True
1,2,11.929,2149.529838,False
2,3,18.971,2981.568587,False
3,4,27.121,4901.206575,False
4,5,33.459,4080.434324,True
5,6,39.862,3134.096190,False
6,7,47.738,4537.185041,False
7,8,54.171,4120.485037,False


In [22]:
events = pd.read_csv("/Users/arnavchouksey/kinetisense/data/synthetic/synthetic_events.csv")
true_throws = events[events["label"] == "throw"]

check = throws_table.copy()
check["true_peak_dps"] = true_throws["true_peak_dps"].values
check #Here is where we check to see if throws_table is equal to the actualy "throws" above our set threshold

,throw_id,peak_time_s,peak_dps,clipped,true_peak_dps
0,1,5.394,4338.251844,True,5021.216869
1,2,11.929,2149.529838,False,2157.502763
2,3,18.971,2981.568587,False,2974.022361
3,4,27.121,4901.206575,False,4907.067178
4,5,33.459,4080.434324,True,5047.055059
5,6,39.862,3134.096190,False,3144.810593
6,7,47.738,4537.185041,False,4555.785182
7,8,54.171,4120.485037,False,4119.960431


In [23]:
check.values #This method turns everything into a readable value format for each column

array([[1, 5.394, 4338.25184445237, True, 5021.216868965675],
       [2, 11.929, 2149.529838295014, False, 2157.502762958452],
       [3, 18.971, 2981.5685866556983, False, 2974.0223611416995],
       [4, 27.121, 4901.206574580212, False, 4907.067178095034],
       [5, 33.459, 4080.4343242118994, True, 5047.055058944163],
       [6, 39.862, 3134.096189756615, False, 3144.8105930672136],
       [7, 47.738, 4537.185041401204, False, 4555.785182238532],
       [8, 54.171, 4120.485036863605, False, 4119.96043052644]],
      dtype=object)

In [24]:
check["true_peak_dps"] = true_throws["true_peak_dps"].values


In [25]:
check["difference"] = check["true_peak_dps"] - check["peak_dps"] #This is the "difference" which is a new value we have added on to the table which tells us the difference between the true_peak_degreees per second whichis the answer key versus out clipped dps based on our materials in hand.
check


,throw_id,peak_time_s,peak_dps,clipped,true_peak_dps,difference
0,1,5.394,4338.251844,True,5021.216869,682.965025
1,2,11.929,2149.529838,False,2157.502763,7.972925
2,3,18.971,2981.568587,False,2974.022361,-7.546226
3,4,27.121,4901.206575,False,4907.067178,5.860604
4,5,33.459,4080.434324,True,5047.055059,966.620735
5,6,39.862,3134.096190,False,3144.810593,10.714403
6,7,47.738,4537.185041,False,4555.785182,18.600141
7,8,54.171,4120.485037,False,4119.960431,-0.524606


In [26]:
throws_table.to_csv("/Users/arnavchouksey/kinetisense/data/synthetic/synthetic_throws.csv", index=False) #Save the throws table
throws_table

,throw_id,peak_time_s,peak_dps,clipped
0,1,5.394,4338.251844,True
1,2,11.929,2149.529838,False
2,3,18.971,2981.568587,False
3,4,27.121,4901.206575,False
4,5,33.459,4080.434324,True
5,6,39.862,3134.096190,False
6,7,47.738,4537.185041,False
7,8,54.171,4120.485037,False


In [27]:
HALF = 500
columns = ["time_s", "gx", "gy", "gz", "ax", "ay", "az", "gyro_mag"]
pieces = []
for _, row in throws_table.iterrows():
    peak_label = (df["time_s"] - row["peak_time_s"]).abs().idxmin()
    piece = df.loc[peak_label - HALF : peak_label + HALF, columns].copy()
    piece.insert(0, "throw_id", int(row["throw_id"]))
    pieces.append(piece)
samples = pd.concat(pieces, ignore_index=True)

In [28]:
samples.shape
samples.groupby("throw_id").size()

throw_id
1    1001
2    1001
3    1001
4    1001
5    1001
6    1001
7    1001
8    1001
dtype: int64

In [29]:
assert len(samples) == 8 * 1001, "wrong total row count" #Helps to tell us that everything should be 1001 rows around everything, 

In [30]:
samples.columns

Index(['throw_id', 'time_s', 'gx', 'gy', 'gz', 'ax', 'ay', 'az', 'gyro_mag'], dtype='str')

In [31]:
assert list(samples.columns)==["throw_id", "time_s", "gx", "gy", "gz", "ax", "ay", "az", "gyro_mag"]

In [32]:
samples.to_csv("/Users/arnavchouksey/kinetisense/data/synthetic/synthetic_samples.csv", index=False) #Saves samples as a new csv file to be called upon in the future


In [33]:
samples #Samples CSV table that we created

,throw_id,time_s,gx,gy,gz,ax,ay,az,gyro_mag
0,1,4.894,31.184145,22.202889,16.814690,0.244725,0.252671,1.292537,41.810919
1,1,4.895,33.863486,22.584627,21.298474,0.227052,0.216663,1.285949,45.939374
2,1,4.896,30.701246,21.107205,20.615228,0.237410,0.207460,1.267881,42.580139
3,1,4.897,30.807360,22.574904,21.644051,0.244251,0.217785,1.277302,43.899711
4,1,4.898,29.315165,26.238000,20.397532,0.239150,0.201661,1.258477,44.315583
...,...,...,...,...,...,...,...,...,...
8003,8,54.667,-28.703709,-20.525794,-11.881939,-0.212796,-0.175726,0.798681,37.234280
8004,8,54.668,-24.019919,-17.515419,-14.944604,-0.189321,-0.168166,0.749845,33.272926
8005,8,54.669,-26.608094,-14.891651,-14.945621,-0.173612,-0.147749,0.735088,33.957672
8006,8,54.670,-25.140952,-9.379535,-11.193148,-0.169733,-0.123781,0.819075,29.074554


In [34]:
assert'window' not in samples.columns #Just checking that there is no windows column in the sample csv

In [35]:
assert (samples.groupby("throw_id").size() == 1001).all(), "each throw should have 1001 rows"